# Setup

In [36]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

In [37]:
data_dir = Path.cwd().resolve().parents[2] / "data" / "doppler_waveform"  # modules/doppler_waveform/scratch -> repo root
waveform_img_path = data_dir / "input" / "waveform_envelope.png"
output_dir = data_dir / "output"

# Define Regions of Interest

In [38]:
# Image ROIs use the same (x0, y0, x1, y1) format as test_ocr.
# The Doppler ROI is limited to the WHITE spectral waveform.
# It excludes the ECG at the bottom and the horizontal zero-velocity line.
# Adjust these values if the Terason layout changes.
REF_W, REF_H = 1024, 768
ROIS = {
    "waveform": (85, 420, 697, 556),
}

THRESHOLD = 20  # brightness cutoff, 0-255; tune for different captures
MEDIAN_KERNEL = 1   # odd window width, smooths the extracted upper edge
MAX_INTERPOLATION_GAP = 5  # only fill missing runs of at most 5 columns

In [39]:
def scaled_roi(img, name):
    h, w = img.shape[:2]
    x0, y0, x1, y1 = ROIS[name]
    sx, sy = w / REF_W, h / REF_H
    return int(x0 * sx), int(y0 * sy), int(x1 * sx), int(y1 * sy)


def crop(img, name):
    x0, y0, x1, y1 = scaled_roi(img, name)
    return img[y0:y1, x0:x1], (x0, y0)

# Waveform and Envelope Extraction

In [40]:
def extract_waveform(img, threshold=THRESHOLD):
    """Return Doppler mask, grayscale ROI, and foreground (x,y) pixels in IMAGE coordinates."""
    roi, (x0, y0) = crop(img, "waveform")
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)

    # The waveform is bright/gray, while the graph background is nearly black.
    _, mask = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)

    # Remove isolated single-pixel speckles without erasing the thin waveform.
    mask = cv2.morphologyEx(
        mask, cv2.MORPH_OPEN, np.ones((2, 2), dtype=np.uint8)
    )

    ys, xs = np.where(mask > 0)
    waveform_coords = np.column_stack((xs + x0, ys + y0)).astype(np.int32)
    return mask, gray, waveform_coords


def extract_envelope(mask, gray, offset, median_kernel=MEDIAN_KERNEL,
                     max_gap=MAX_INTERPOLATION_GAP):
    """Extract the intensity-weighted average y-position in original-image coordinates.

    Interpolate ONLY missing runs up to `max_gap` columns, bounded on
    BOTH sides by detected waveform pixels. Larger gaps and missing
    columns at the image edges remain NaN (not invented measurements).

    Parameters
    ----------
    mask : (H, W) uint8 binary mask of foreground Doppler pixels
    gray : (H, W) uint8 grayscale ROI matching `mask` dimensions
    offset : (x0, y0) top-left ROI offset in original image coordinates
    median_kernel : int, odd window size for median smoothing (1 to disable)
    max_gap : int, maximum missing column run length to linearly interpolate

    Returns
    -------
    envelope_coords : (W, 2) x,y image coordinates, with NaN for large gaps
    valid : (W,) True where the waveform was actually detected (not filled)
    """
    if mask.shape != gray.shape:
        raise ValueError(f"mask shape {mask.shape} does not match gray shape {gray.shape}")

    x0, y0 = offset
    _, width = mask.shape
    envelope_y = np.full(width, np.nan, dtype=float)

    for x in range(width):
        foreground_y = np.flatnonzero(mask[:, x] > 0)
        if foreground_y.size:
            weights = gray[foreground_y, x].astype(float)
            weight_sum = weights.sum()
            if weight_sum > 0:
                weighted_y = np.dot(foreground_y, weights) / weight_sum
                envelope_y[x] = weighted_y + y0

    valid = np.isfinite(envelope_y)
    if not np.any(valid):
        raise ValueError("No Doppler waveform detected: check ROI and threshold.")

    # Fill only short INTERNAL gaps. Do not extend the first/last observation.
    valid_indices = np.flatnonzero(valid)
    for left, right in zip(valid_indices[:-1], valid_indices[1:]):
        gap_length = right - left - 1
        if 0 < gap_length <= max_gap:
            envelope_y[left + 1:right] = np.linspace(
                envelope_y[left], envelope_y[right], gap_length + 2
            )[1:-1]

    # Optional median smoothing, independently within each continuous run.
    # A median filter must never smooth through a long missing (NaN) gap.
    if median_kernel > 1:
        if median_kernel % 2 != 1:
            raise ValueError("median_kernel must be odd")
        finite = np.isfinite(envelope_y)
        edges = np.diff(np.r_[False, finite, False].astype(int))
        for run_start, run_end in zip(np.flatnonzero(edges == 1),
                                      np.flatnonzero(edges == -1)):
            segment = envelope_y[run_start:run_end]
            pad = median_kernel // 2
            padded = np.pad(segment, pad, mode="edge")
            windows = np.lib.stride_tricks.sliding_window_view(padded, median_kernel)
            envelope_y[run_start:run_end] = np.median(windows, axis=1)

    envelope_x = np.arange(width) + x0
    envelope_coords = np.column_stack((envelope_x, envelope_y))
    return envelope_coords, valid

# Main Function

In [41]:
IMAGE_EXTS = {".png", ".jpg", ".jpeg"}


def process_image(input_path, output_path, show_plots=False):
    """Extract the Doppler waveform and its upper envelope from a single image.

    Parameters
    ----------
    input_path : str or Path
        Input Terason screenshot.
    output_path : str or Path
        Folder where coordinate CSVs, mask, and overlay are saved.
    show_plots : bool, default False
        Whether to display interactive matplotlib windows.

    Returns
    -------
    dict
        waveform_coords: (N, 2) pixel [x, y] pairs
        envelope_coords: (W, 2) pixel [x, y] pairs
        envelope_y:      (W,) y pixel positions only
        waveform_mask:   2D binary crop mask
        valid_columns:   (W,) whether a waveform pixel was found in each column
        interpolated_columns: (W,) short-gap pixels filled by interpolation
    """
    input_path = Path(input_path)
    output_path = Path(output_path)
    img = cv2.imread(str(input_path))
    if img is None:
        raise FileNotFoundError(f"Could not read image: {input_path}")

    mask, gray, waveform_coords = extract_waveform(img)
    _, offset = crop(img, "waveform")
    envelope_coords, valid = extract_envelope(mask, gray, offset)
    envelope_y = envelope_coords[:, 1]
    interpolated = np.isfinite(envelope_y) & ~valid

    output_path.mkdir(parents=True, exist_ok=True)
    np.savetxt(
        output_path / "waveform_coordinates.csv", waveform_coords,
        delimiter=",", header="x_pixel,y_pixel", comments="", fmt="%d"
    )
    np.savetxt(
        output_path / "envelope_coordinates.csv", envelope_coords,
        delimiter=",", header="x_pixel,y_pixel", comments="", fmt=["%d", "%.3f"]
    )
    np.savetxt(
        output_path / "envelope_y.csv", envelope_y,
        delimiter=",", header="y_pixel", comments="", fmt="%.3f"
    )
    cv2.imwrite(str(output_path / "waveform_mask.png"), mask)

    # Overlay in ORIGINAL image coordinates; no physical-unit conversion yet.
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.plot(envelope_coords[:, 0], envelope_y, color="red", linewidth=1.5,
            label="Extracted envelope")
    x0, y0, x1, y1 = scaled_roi(img, "waveform")
    ax.set_xlim(x0 - 10, x1 + 10)
    ax.set_ylim(y1 + 15, y0 - 15)
    ax.set_title("Terason spectral Doppler: extracted envelope")
    ax.set_xlabel("X coordinate (pixels)")
    ax.set_ylabel("Y coordinate (pixels)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_path / "envelope_overlay.png", dpi=160)
    if show_plots:
        plt.show()
    plt.close(fig)

    print(f"Saved files to: {output_path.resolve()}")
    print(f"Waveform pixels: {len(waveform_coords)}")
    print(f"Envelope samples: {len(envelope_y)}")
    print(f"Columns with a detected signal: {valid.sum()} / {len(valid)}")
    print(f"Short-gap interpolated columns: {interpolated.sum()}")
    print(f"Long-gap/edge missing columns: {np.isnan(envelope_y).sum()}")

    return {
        "waveform_coords": waveform_coords,
        "envelope_coords": envelope_coords,
        "envelope_y": envelope_y,
        "waveform_mask": mask,
        "valid_columns": valid,
        "interpolated_columns": interpolated,
    }


def main(input_path, output_path, show_plots=False):
    """Run Doppler waveform extraction on either a single image or a directory of images."""
    input_path, output_path = Path(input_path), Path(output_path)

    if input_path.is_file():
        return process_image(input_path, output_path, show_plots=show_plots)

    if input_path.is_dir():
        images = sorted(
            p for p in input_path.iterdir()
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS
        )
        if not images:
            raise FileNotFoundError(f"No images found in directory: {input_path}")

        results = {}
        for img_path in images:
            print(f"\nProcessing {img_path.name}...")
            results[img_path.name] = process_image(
                img_path,
                output_path / img_path.stem,
                show_plots=show_plots,
            )
        return results

    raise FileNotFoundError(f"Input path does not exist: {input_path}")

# Test Waveform and Envelope

In [47]:
results = main("../../../data/doppler_waveform/input", "../../../data/doppler_waveform/output")


Processing waveform_bad_signal.png...
Saved files to: /Users/ursulachui/Documents/School/Capstone-Throwaway-Tests/data/doppler_waveform/output/waveform_bad_signal
Waveform pixels: 10050
Envelope samples: 612
Columns with a detected signal: 531 / 612
Short-gap interpolated columns: 14
Long-gap/edge missing columns: 67

Processing waveform_bad_streaks.png...
Saved files to: /Users/ursulachui/Documents/School/Capstone-Throwaway-Tests/data/doppler_waveform/output/waveform_bad_streaks
Waveform pixels: 9873
Envelope samples: 612
Columns with a detected signal: 444 / 612
Short-gap interpolated columns: 23
Long-gap/edge missing columns: 145

Processing waveform_good.png...
Saved files to: /Users/ursulachui/Documents/School/Capstone-Throwaway-Tests/data/doppler_waveform/output/waveform_good
Waveform pixels: 20443
Envelope samples: 612
Columns with a detected signal: 612 / 612
Short-gap interpolated columns: 0
Long-gap/edge missing columns: 0

Processing waveform_sweepbar.png...
Saved files to:

# END